# Chess Numba — Match-SPSA Colab Runner（含 Google Drive 紀錄與 `constants.py` 自動歸檔）

本 Notebook 在 Google Colab 執行 `tune_eval_match/spsa_match.py`（單進程共享 JIT 對局式 SPSA）與 `tools/run_eval_param_match.py`（400k nodes 獨立 SPRT 驗證），適合調優**威脅表（`threats_mg` / `threats_eg`）**、王安全、空間與動態通路兵等非靜態評估權重。

### 使用前提醒
1. **同步程式碼**：Colab 會從 GitHub `origin/main` clone 專案，開跑前請先在本機執行 `git push origin main`（或將打包好的專案壓縮檔路徑填在 `DRIVE_REPO_ZIP`）。
2. **Google Drive 持久化**：每完成一個 iteration 與每次 L2 驗收，會即時同步以下檔案至 `MyDrive/chess_numba_data/eval_match_spsa/<RUN_TAG>/`：
   * `spsa_log.jsonl`、`spsa_stdout.log`、`checkpoint.json`、`latest_params.json`、`best_params.json`、`best_constants_diff.txt`、`run_meta.json`
   * `constants_exports/constants_best.py`、`constants_exports/constants_latest.py`、`constants_exports/tuned_constants_best.py`（可直接帶回本機覆蓋 `chess_engine/classical/constants.py`）
   * Stage 3 驗證輸出：`constants_exports/verify_400k.json`、`constants_exports/verify_400k.log`
3. **建議順序**：先跑 `BLOCK = "threats_mg"`（17 維）並通過 400k SPRT，寫回 `constants.py` 後再調 `BLOCK = "threats_eg"`。


In [ ]:
# 1. 檢視 Colab 硬體與 CPU 規格
import os
import subprocess
import sys

subprocess.run(["bash", "-lc", "lscpu | grep -E 'Model name|Socket|Thread|NUMA|CPU\\(s\\)' || nproc"], check=False)
subprocess.run(["free", "-h"], check=False)
print("Python:", sys.version)
print("os.cpu_count():", os.cpu_count())

In [ ]:
# 2. 實驗參數與 Google Drive 路徑設定（續跑時請維持相同 RUN_TAG / BLOCK）
from pathlib import Path

# --- GitHub / 來源設定 ---
REPO_URL = "https://github.com/b08203031/chess_numba.git"
BRANCH = "main"
# 選填：若尚未 git push，可將打包好的 repo zip 放上 Drive 並填路徑，例如 "/content/drive/MyDrive/chess_numba_sync.zip"
DRIVE_REPO_ZIP = ""
# 選填：若要基於先前調好的 constants_best.py 繼續調下一個 block（例如 threats_mg -> threats_eg），填其 Drive 路徑
BASE_CONSTANTS_OVERRIDE = ""

# --- 調參目標設定 ---
RUN_TAG = "spsa_eval_threats_mg"
BLOCK = "threats_mg"          # 可選: threats_mg, threats_eg, threats, king_weights, g4, a, space, passed, passed_rank, material, imbalance
CUSTOM_PARAMS = ""            # 選填逗號分隔清單（非空時會覆蓋 BLOCK）

# --- L1 / L2 Match-SPSA 超參數 ---
NODES = 80000
GAMES = 500                   # 掛機時間短可改 300；時間充裕維持 500 梯度更穩
ITERS = 150                   # 0 = 不限輪數直到觸發 --l2-stop-after-fails 或手動停止
SPSA_A = 25.0
TT_MB = 8
CONCURRENCY = 0               # 0 = 自動依 CPU 核心數設定（建議上限 8）

L2_EVERY = 10
L2_NODES = 200000
L2_GAMES = 500
L2_CANDIDATE = "tail"         # tail（窗口後半平均，推薦）或 latest
L2_MIN_SCORE = 0.510
L2_SPRT = ""                  # 選填，例如 "0,10"；空字串則使用 L2_MIN_SCORE 門檻
L2_STOP_AFTER_FAILS = 3       # 連續 L2 未過幾次後自動停止（0 = 不自動停）

# --- 接續與持久化設定 ---
RESUME = True                 # True = 若 Drive 已有 checkpoint.json 則自動 --resume
USE_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/chess_numba_data/eval_match_spsa")
LOCAL_ROOT = Path("/content/eval_match_runs")

DRIVE_RUN_DIR = DRIVE_ROOT / RUN_TAG
DRIVE_RESULTS_DIR = DRIVE_RUN_DIR / "constants_exports"
LOCAL_RUN_DIR = LOCAL_ROOT / RUN_TAG

print(f"RUN_TAG={RUN_TAG} | BLOCK={BLOCK} | RESUME={RESUME}")
print(f"DRIVE_RUN_DIR={DRIVE_RUN_DIR}")
print(f"DRIVE_RESULTS_DIR={DRIVE_RESULTS_DIR}")

In [ ]:
# 3. 掛載 Google Drive、同步專案並準備工作目錄
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = ("google.colab" in sys.modules) or (os.name != "nt" and Path("/content").is_dir())
WORKDIR = Path("/content/chess_numba") if IN_COLAB else Path.cwd()

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if IN_COLAB:
    if DRIVE_REPO_ZIP and Path(DRIVE_REPO_ZIP).is_file():
        print(f"[Repo] Extracting {DRIVE_REPO_ZIP} -> {WORKDIR}")
        if WORKDIR.exists():
            shutil.rmtree(WORKDIR)
        WORKDIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DRIVE_REPO_ZIP, "r") as zf:
            zf.extractall(WORKDIR)
    else:
        if not (WORKDIR / ".git").is_dir():
            print(f"[Git] Cloning {REPO_URL} (branch={BRANCH}) -> {WORKDIR}")
            subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
        else:
            print(f"[Git] Fetching origin/{BRANCH} in {WORKDIR}")
            subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin"], check=True)
            subprocess.run(["git", "-C", str(WORKDIR), "checkout", "-f", BRANCH], check=True)
            subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)

    os.chdir(WORKDIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba", "numpy", "chess", "matplotlib"], check=True)
else:
    DRIVE_RUN_DIR = Path("tournament_analysis") / RUN_TAG
    DRIVE_RESULTS_DIR = DRIVE_RUN_DIR / "constants_exports"
    LOCAL_RUN_DIR = DRIVE_RUN_DIR

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

if BASE_CONSTANTS_OVERRIDE and Path(BASE_CONSTANTS_OVERRIDE).is_file():
    target_c = WORKDIR / "chess_engine" / "classical" / "constants.py"
    shutil.copy2(BASE_CONSTANTS_OVERRIDE, target_c)
    print(f"[Override] Applied {BASE_CONSTANTS_OVERRIDE} -> {target_c}")

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

# 若啟用 RESUME，先將 Drive 上既有的 checkpoint 與紀錄拉回本地快速工作目錄
if RESUME and LOCAL_RUN_DIR != DRIVE_RUN_DIR:
    for fname in (
        "checkpoint.json",
        "latest_params.json",
        "best_params.json",
        "final_params.json",
        "run_meta.json",
        "spsa_log.jsonl",
        "best_constants_diff.txt",
    ):
        src = DRIVE_RUN_DIR / fname
        if src.exists():
            shutil.copy2(src, LOCAL_RUN_DIR / fname)

print("WORKDIR:", WORKDIR)
subprocess.run(["git", "-C", str(WORKDIR), "log", "-1", "--oneline"], check=False)

In [ ]:
# 4. 起跑前檢查：顯示目標 Block 參數預設值與現有斷點狀態
import json
from tune_eval_match.param_config import bounds, default_params, select_custom_params, set_active_block

active = select_custom_params(CUSTOM_PARAMS.split(",")) if CUSTOM_PARAMS.strip() else set_active_block(BLOCK)
cur_defaults = default_params()
print(f"Active block: {active} ({len(cur_defaults)} params)")
for k, v in cur_defaults.items():
    lo, hi = bounds(k)
    print(f"  {k:32s} cur={int(v):6d}  bounds=[{lo:g}, {hi:g}]")

ckpt_path = DRIVE_RUN_DIR / "checkpoint.json"
if ckpt_path.exists():
    ckpt = json.loads(ckpt_path.read_text(encoding="utf-8"))
    print(f"\n[Drive Checkpoint] iter={ckpt.get('iter')} | best_l2_score={ckpt.get('best_l2_score')} | opening_offset={ckpt.get('opening_offset')}")
else:
    print("\n[Drive Checkpoint] 尚無斷點，將從 constants.py 現行值起跑。")

In [ ]:
# 5. 定義 Drive 即時同步與 constants.py 匯出函式，並啟動 Match-SPSA
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from tune_eval_match.eval_weights import format_constants_diff
from tune_eval_match.export_constants import export_params_to_files, load_params_dict

SYNC_FILES = (
    "spsa_log.jsonl",
    "checkpoint.json",
    "latest_params.json",
    "best_params.json",
    "final_params.json",
    "run_meta.json",
    "best_constants_diff.txt",
)


def atomic_copy(src: Path, dst: Path) -> None:
    if not src.exists() or src.resolve() == dst.resolve():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)


def sync_run_to_drive(export_consts: bool = True) -> None:
    for fname in SYNC_FILES:
        atomic_copy(LOCAL_RUN_DIR / fname, DRIVE_RUN_DIR / fname)
    if not export_consts:
        return
    for tag, json_name in (("best", "best_params.json"), ("latest", "latest_params.json")):
        pjson = LOCAL_RUN_DIR / json_name
        if not pjson.exists():
            continue
        params = load_params_dict(pjson)
        export_params_to_files(
            params,
            target_path=WORKDIR / "chess_engine" / "classical" / "constants.py",
            out_constants=DRIVE_RESULTS_DIR / f"constants_{tag}.py",
            out_snippet=DRIVE_RESULTS_DIR / f"tuned_constants_{tag}.py",
            apply_inplace=False,
            verbose=False,
        )
        diff_txt = format_constants_diff(params)
        (DRIVE_RESULTS_DIR / f"constants_diff_{tag}.txt").write_text(diff_txt + "\n", encoding="utf-8")


cpus = os.cpu_count() or 2
eff_c = CONCURRENCY if CONCURRENCY > 0 else max(1, min(8, cpus - 1 if cpus > 2 else cpus))

cmd = [
    sys.executable, "-u", "tune_eval_match/spsa_match.py",
    "--nodes", str(NODES),
    "--games", str(GAMES),
    "--iters", str(ITERS),
    "--A", str(SPSA_A),
    "--tt-mb", str(TT_MB),
    "-c", str(eff_c),
    "--l2-every", str(L2_EVERY),
    "--l2-nodes", str(L2_NODES),
    "--l2-games", str(L2_GAMES),
    "--l2-candidate", str(L2_CANDIDATE),
    "--l2-min-score", str(L2_MIN_SCORE),
    "--l2-stop-after-fails", str(L2_STOP_AFTER_FAILS),
    "--out-dir", str(LOCAL_RUN_DIR),
]
if CUSTOM_PARAMS.strip():
    cmd.extend(["--params", CUSTOM_PARAMS.strip()])
else:
    cmd.extend(["--block", BLOCK])
if L2_SPRT.strip():
    cmd.extend(["--l2-sprt", L2_SPRT.strip()])
local_ckpt = LOCAL_RUN_DIR / "checkpoint.json"
if RESUME and local_ckpt.exists():
    cmd.extend(["--resume", str(local_ckpt)])

print("[Launch]", " ".join(cmd), flush=True)
stdout_log = DRIVE_RUN_DIR / "spsa_stdout.log"
t_start = time.time()

proc = subprocess.Popen(
    cmd,
    cwd=str(WORKDIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    with stdout_log.open("a", encoding="utf-8", newline="\n") as lf:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            lf.write(line)
            if line.startswith("[iter ") or "--- L2" in line or "[L2" in line or "[SPSA]" in line:
                lf.flush()
                sync_run_to_drive(export_consts=True)
    rc = proc.wait()
    print(f"\n[Done] exit_code={rc} elapsed={(time.time() - t_start)/60:.1f} min")
except KeyboardInterrupt:
    print("\n[Interrupted] Terminating spsa_match.py and syncing latest state to Drive...")
    proc.terminate()
    proc.wait(timeout=15)
finally:
    sync_run_to_drive(export_consts=True)
    print(f"[Drive Synced] {DRIVE_RUN_DIR} & {DRIVE_RESULTS_DIR}")

In [ ]:
# 6. 檢視 SPSA 進度曲線、top drift_z 與 best_constants_diff
import json
import matplotlib.pyplot as plt

log_file = DRIVE_RUN_DIR / "spsa_log.jsonl"
if not log_file.exists():
    print("尚無 spsa_log.jsonl:", log_file)
else:
    l1_rows, l2_rows = [], []
    for raw in log_file.read_text(encoding="utf-8").splitlines():
        if not raw.strip():
            continue
        row = json.loads(raw)
        if row.get("type") == "l2_validate":
            l2_rows.append(row)
        elif "iter" in row and "score_plus" in row:
            l1_rows.append(row)

    print(f"L1 iterations={len(l1_rows)} | L2 validations={len(l2_rows)}")
    if l1_rows:
        last = l1_rows[-1]
        top_z = sorted(last.get("drift_z", {}).items(), key=lambda kv: abs(kv[1]), reverse=True)[:8]
        print("最新 L1 drift_z Top 8:", ", ".join(f"{k}:{v:+.2f}" for k, v in top_z))

    diff_file = DRIVE_RUN_DIR / "best_constants_diff.txt"
    if diff_file.exists():
        print("\n=== best_constants_diff.txt ===")
        print(diff_file.read_text(encoding="utf-8"))

    if l1_rows:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        xs = [r["iter"] for r in l1_rows]
        ys = [r["elo_plus_vs_minus"] for r in l1_rows]
        axes[0].plot(xs, ys, marker=".", alpha=0.8)
        axes[0].axhline(0.0, color="gray", linestyle="--", linewidth=1)
        axes[0].set_title(f"{RUN_TAG} L1 Elo(+ vs -)")
        axes[0].set_xlabel("iter")
        axes[0].set_ylabel("Elo")

        if l2_rows:
            l2_x = [r["iter"] for r in l2_rows]
            l2_y = [r["elo"] for r in l2_rows]
            colors = ["green" if r.get("accepted") else "red" for r in l2_rows]
            axes[1].scatter(l2_x, l2_y, c=colors)
            axes[1].axhline(0.0, color="gray", linestyle="--", linewidth=1)
            axes[1].set_title("L2 Validation Elo vs Best (green=accepted)")
            axes[1].set_xlabel("iter")
            axes[1].set_ylabel("Elo")
        else:
            axes[1].set_title("No L2 validations yet")

        plt.tight_layout()
        png_path = DRIVE_RUN_DIR / "spsa_progress.png"
        plt.savefig(png_path, dpi=140)
        plt.show()
        print("Saved plot:", png_path)

In [ ]:
# 7. Stage 3 — 400k nodes 獨立 SPRT 最終確認（結果直接寫入 Google Drive）
import os
import subprocess
import sys

VERIFY_SOURCE = "best"        # "best" 使用 best_params.json；若無則可改 "latest"
VERIFY_NODES = 400000
VERIFY_GAMES = 2000
VERIFY_SPRT = "0,10"          # 亦可改 "off" 跑滿 VERIFY_GAMES 局
VERIFY_BATCH_GAMES = 50
COMPARE_BASELINE_OLD = False  # False=對戰現行 classical；True=對戰 classical_old

cand_json = DRIVE_RUN_DIR / ("best_params.json" if VERIFY_SOURCE == "best" else "latest_params.json")
if not cand_json.exists():
    raise FileNotFoundError(f"找不到候選參數檔: {cand_json}")

cpus = os.cpu_count() or 2
eff_c = CONCURRENCY if CONCURRENCY > 0 else max(1, min(8, cpus - 1 if cpus > 2 else cpus))
verify_json = DRIVE_RESULTS_DIR / "verify_400k.json"
verify_log = DRIVE_RESULTS_DIR / "verify_400k.log"

v_cmd = [
    sys.executable, "-u", "tools/run_eval_param_match.py",
    "--params-json", str(cand_json),
    "--nodes", str(VERIFY_NODES),
    "--games", str(VERIFY_GAMES),
    "--sprt", str(VERIFY_SPRT),
    "--batch-games", str(VERIFY_BATCH_GAMES),
    "-c", str(eff_c),
    "--out", str(verify_json),
]
if COMPARE_BASELINE_OLD:
    v_cmd.append("--baseline-old")

print("[Verify Launch]", " ".join(v_cmd), flush=True)
v_proc = subprocess.Popen(v_cmd, cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with verify_log.open("w", encoding="utf-8", newline="\n") as lf:
    assert v_proc.stdout is not None
    for line in v_proc.stdout:
        print(line, end="", flush=True)
        lf.write(line)
v_rc = v_proc.wait()
print(f"[Verify Done] exit_code={v_rc} | saved: {verify_json}")

In [ ]:
# 8. 匯出最終 constants.py 至 Google Drive（並顯示摘要）
from tune_eval_match.eval_weights import format_constants_diff
from tune_eval_match.export_constants import export_params_to_files, load_params_dict

APPLY_TO_LOCAL_COLAB_REPO = False  # True 時會一併覆蓋 Colab /content/chess_numba/chess_engine/classical/constants.py

final_json = DRIVE_RUN_DIR / "best_params.json"
if not final_json.exists():
    final_json = DRIVE_RUN_DIR / "latest_params.json"
if not final_json.exists():
    raise FileNotFoundError("找不到 best_params.json 或 latest_params.json")

params = load_params_dict(final_json)
out_c = DRIVE_RESULTS_DIR / "constants_final.py"
out_s = DRIVE_RESULTS_DIR / "tuned_constants_final.py"
changed = export_params_to_files(
    params,
    target_path=WORKDIR / "chess_engine" / "classical" / "constants.py",
    out_constants=out_c,
    out_snippet=out_s,
    apply_inplace=APPLY_TO_LOCAL_COLAB_REPO,
    verbose=True,
)
(DRIVE_RESULTS_DIR / "constants_diff_final.txt").write_text(format_constants_diff(params) + "\n", encoding="utf-8")

print("\n=== Google Drive 歸檔產物 ===")
for p in sorted(DRIVE_RESULTS_DIR.glob("*")):
    print(" ", p)
print("\n=== 參數差異摘要 ===")
print(format_constants_diff(params))